In [9]:
import numpy as np
import pandas as pd
from datetime import datetime

# ==============================
# 1️⃣ 예측값 불러오기
# ==============================
final_test = np.load("./catboost_artifacts/test_pred_0.5.npy")  # shape (len(test_df),)

# ==============================
# 2️⃣ 샘플 제출 파일 불러오기
# ==============================
df = pd.read_csv("./open/sample_submission.csv")  # id와 pred 컬럼이 있다고 가정

# ==============================
# 4️⃣ CSV 저장
# ==============================
output_file = f"OSM_nodrop_{datetime.now().strftime('%m%d-%H%M')}.csv"
df.to_csv(output_file, index=False)

print("CSV 제출 파일 저장 완료:", output_file)
print("final_test shape:", final_test.shape)
print("submission df shape:", df.shape)
print("예측값 범위:", final_test.min(), final_test.max())


CSV 제출 파일 저장 완료: OSM_nodrop_0211-2344.csv
final_test shape: (90067,)
submission df shape: (90067, 2)
예측값 범위: 0.000260216595092404 0.770401209543913


In [11]:
import numpy as np
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, precision_score, recall_score, confusion_matrix

# 저장된 OOF 불러오기
oof_proba = np.load("./catboost_artifacts/oof_pred_0.5.npy")  # 파일명 확인
y_true = np.load("y_tabnet.npy", allow_pickle=True)        # 실제 타겟

# Threshold 설정 (보통 0.5)
TH = 0.5
oof_pred = (oof_proba > TH).astype(int)

# 평가 지표 계산
print("Confusion Matrix:\n", confusion_matrix(y_true, oof_pred))
print("Accuracy:", accuracy_score(y_true, oof_pred))
print("Precision:", precision_score(y_true, oof_pred))
print("Recall:", recall_score(y_true, oof_pred))
print("F1 Score:", f1_score(y_true, oof_pred))
print("ROC AUC:", roc_auc_score(y_true, oof_proba))


Confusion Matrix:
 [[183667   6456]
 [ 58611   7617]]
Accuracy: 0.7461800422077542
Precision: 0.5412492005968876
Recall: 0.11501177749592317
F1 Score: 0.1897112115664811
ROC AUC: 0.7402086781976632


In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'AppleGothic'
matplotlib.rcParams['axes.unicode_minus'] = False
import os
from sklearn.metrics import roc_curve, auc, ConfusionMatrixDisplay

SAVE_DIR = os.path.join('..', 'outputs', 'figures')
os.makedirs(SAVE_DIR, exist_ok=True)

# ==============================
# 1. Feature Importance (Top 20)
# ==============================
from catboost import CatBoostClassifier, Pool
import glob

model_path = sorted(glob.glob('./models/IVF_Fold*.cbm'))[-1]
model = CatBoostClassifier()
model.load_model(model_path)

fi = model.get_feature_importance()
fn = model.feature_names_

import pandas as pd
fi_df = pd.DataFrame({'feature': fn, 'importance': fi})
fi_df = fi_df.sort_values('importance', ascending=True).tail(20)

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(fi_df['feature'], fi_df['importance'], color='#2196F3')
ax.set_xlabel('Importance')
ax.set_title('Top 20 Feature Importance (CatBoost)')
plt.tight_layout()
fig.savefig(os.path.join(SAVE_DIR, 'feature_importance.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✅ feature_importance.png 저장 완료')

In [ ]:
# ==============================
# 2. ROC Curve
# ==============================
fpr, tpr, _ = roc_curve(y_true, oof_proba)
roc_auc_val = auc(fpr, tpr)

fig, ax = plt.subplots(figsize=(8, 8))
ax.plot(fpr, tpr, color='#FF5722', lw=2, label=f'ROC curve (AUC = {roc_auc_val:.4f})')
ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve')
ax.legend(loc='lower right')
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.05])
plt.tight_layout()
fig.savefig(os.path.join(SAVE_DIR, 'roc_curve.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✅ roc_curve.png 저장 완료')

In [ ]:
# ==============================
# 3. Confusion Matrix
# ==============================
cm = confusion_matrix(y_true, oof_pred)

fig, ax = plt.subplots(figsize=(7, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Negative', 'Positive'])
disp.plot(ax=ax, cmap='Blues', values_format='d')
ax.set_title('Confusion Matrix')
plt.tight_layout()
fig.savefig(os.path.join(SAVE_DIR, 'confusion_matrix.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✅ confusion_matrix.png 저장 완료')